# MINATO observing planning

## Goal

Build evenly spaced orbital-phase windows, assess whether a target is observable, and inspect its airmass through one night with `minato.observing`.


## Setup

This tutorial requires the normal MINATO dependencies, including `astroplan`. It uses explicit observatory coordinates and disables Astropy network downloads, so it does not depend on the online site or IERS registries.


In [ ]:
import astropy.units as u
import matplotlib.pyplot as plt
from astropy.coordinates import EarthLocation
from astropy.utils import iers

from minato.observing import (
    compute_phases,
    generate_phase_windows,
    plot_night_visibility,
)

iers.conf.auto_download = False
iers.conf.auto_max_age = None
location = EarthLocation.from_geodetic(
    lon=-17.89 * u.deg,
    lat=28.76 * u.deg,
    height=2300 * u.m,
)


## Steps

### 1. Generate phase windows

Start with the phase geometry alone. A tolerance of `0.1` means each half-window is 10% of the spacing between requested phases.


In [ ]:
windows = generate_phase_windows(
    first_time="2026-07-20 20:00",
    period_days=3.2,
    num_epochs=8,
    phase_tolerance=0.1,
    max_time="2026-07-24 20:00",
)
windows.head(8)


### 2. Apply observability constraints

`compute_phases` combines the same phase grid with altitude, airmass, and twilight constraints. It returns a DataFrame; printing and CSV output are optional.


In [ ]:
assessed = compute_phases(
    time="2026-07-20 20:00",
    location=location,
    period=3.2,
    num_epochs=8,
    phase_tolerance=0.1,
    max_date="2026-07-24 20:00",
    ra="18h09m17.69s",
    dec="-23d59m18.23s",
    name="Example target",
    twilight="nautical",
    alt_min=20,
    airmass_max=2.5,
    print_results=False,
)
assessed[["phase", "nominal_utc", "observable"]].head(8)


### 3. Plot one night

The plotting API returns the Matplotlib objects and transformed coordinates. It does not display or save automatically.


In [ ]:
figure, axes, coordinates = plot_night_visibility(
    names="Example target",
    ra="18h09m17.69s",
    dec="-23d59m18.23s",
    time="2026-07-20 12:00",
    location=location,
    show_moon=False,
)
axes.set_title("Example target night visibility")
plt.show()


## Checks

These checks confirm that the phase grid is ordered, the observability result is complete, and the plotted target was returned.


In [ ]:
assert windows["nominal_mjd"].is_monotonic_increasing
assert assessed["observable"].notna().all()
assert "Example target" in coordinates
print("Observing tutorial checks passed.")


## Next steps

Replace the example target, observatory coordinates, period, and date range with your programme values. For saved schedules, pass `output_path` to `compute_phases`; existing files are protected unless `overwrite=True`.
